# Radial Basis Nearest Neighbor (RBNN)

- Giulia Monteiro Garrido (RA: 24010281)
- Mateus Antezana da Silva (RA: )
- Vitor Furuta da Silva (RA: 24008775)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Sequence
from functools import wraps

In [ ]:
type numero = int | float

In [2]:
def tratamento_erros(funcao):
    @wraps(funcao)
    def funcao_interna(*args, **kwargs):
        try:
            return funcao(*args, **kwargs)
        except Exception as e:
            print(f"Erro na função {funcao_interna.__name__}\nTipo: {e}")
            return None
    return funcao_interna

In [19]:
@tratamento_erros
class Distancias:
    def __init__(self)->None:
        pass

    def _validar_vetores(self, A:Sequence[numero], B:Sequence[numero] | numero) -> np.asarray[float] | None:
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)

        if isinstance(B, list) and A.shape != B.shape:
            raise ValueError("Os vetores devem ter a mesma dimensão.")

        if A.size == 0:
            raise ValueError("Os vetores não podem estar vazios.")
        return A, B

    def minkowski(self, A:Sequence[numero], B:Sequence[numero] | numero, p:int) -> numero | None:
        if p <= 0:
            raise ValueError("p deve ser maior que zero.")
        A, B = self._validar_vetores(A, B) 
        return sum(abs(A[i]-(B[i] if(isinstance(B, list)) else B))**p for i in range(len(A))) ** (1/p)
        
    def cosseno(self, A:Sequence[numero], B:Sequence[numero]) -> numero | None:
        if(not isinstance(B, list)):
            print("B precisa ser uma lista")
            return
        A, B = self._validar_vetores(A, B)

        produto = np.dot(A, B)
        norma_A = np.linalg.norm(A)
        norma_B = np.linalg.norm(B)

        if norma_A == 0 or norma_B == 0:
            return 0
        return float(1 - (produto/ (norma_A * norma_B)))

    def manhattan(self, A:Sequence[numero], B:Sequence[numero]) -> numero | None:
        A, B = self._validar_vetores(A, B)
        return sum((abs(A[i]-(B[i] if(isinstance(B, list)) else B ))) for i in range(len(A)))

    def euclidiana(self, A:Sequence[numero], B:Sequence[numero]) -> numero | None:
        A, B = self._validar_vetores(A, B)
        return sum(((A[i]-(B[i] if isinstance(B, list) else B))**2) for i in range(len(A)))**(1/2)

In [23]:
def pesos(x, X_treino, sigma, distancia)->int | float: # gaussiana
    distancias = np.array([
        distancia(x, xi) for xi in X_treino
    ])
    formula = 1 / (sigma * np.sqrt(2 * np.pi))
    expoente = (distancias ** 2) / (2 * sigma ** 2)
    pesos = formula * np.exp(-expoente)
    return pesos


def regressao(x, X_treino, y_treino, sigma, distancia)->int | float:

    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:

        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    return np.sum(w * y_treino) / np.sum(w)

def classificacao(x, X_treino, y_treino, sigma, distancia):

    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:

        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    y_treino = np.asarray(y_treino)
    classes = np.unique(y_treino)
    votos = {c: w[y_treino == c].sum() for c in classes}
    return max(votos, key=votos.get)


In [24]:
@tratamento_erros
class RBNNRegressor:
    def __init__(self, sigma: int | float, distancia:int | float)->None:
        self.sigma = sigma
        self.distancia = distancia

    def fit(self, X_treino, y_treino)->None:
        self.X_treino = np.asarray(X_treino)
        self.y_treino = np.asarray(y_treino)

    def predict(self, X_teste):
        return np.array([
            regressao(x, self.X_treino, self.y_treino, self.sigma, self.distancia)
            for x in X_teste
        ])

In [25]:
@tratamento_erros
class RBNNClassifier:
    def __init__(self, sigma: float, distancia):
        self.sigma = sigma
        self.distancia = distancia

    def fit(self, X_treino:Sequence[int|float], y_treino:Sequence[int|float])->None:
        self.X_treino = np.asarray(X_treino)
        self.y_treino = np.asarray(y_treino)

    def predict(self, X_teste)->Sequence[int|float]:
        return np.array([
            classificacao(x, self.X_treino, self.y_treino, self.sigma, self.distancia)
            for x in X_teste
        ])

## Carregando bases

### IRIS

In [8]:
iris = sns.load_dataset("iris")

X_iris = iris.drop(columns="species").to_numpy()
y_iris = iris["species"].to_numpy()

print(iris.shape)          # (150, 5)
iris.head()


(150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


### Abalone

In [ ]:
# !pip install ucimlrepo instalar esse pip

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X =  pd.DataFrame(abalone.data.features)
y =  pd.DataFrame(abalone.data.targets)